# 04. Giao thức serial với firmware ảo

Notebook này cho cả hai đứa xem app gửi gì và thiết bị trả lời gì theo `docs/PROTOCOL.md`, không cần board. Thiết bị ở đây là `app.fake_device.FakeDevice`, còn phía app dùng `app.serial_client.SerialClient`, chính là phần code sẽ dùng khi nối board thật.

Firmware thật nên trả lời giống hệt bảng ở mục 1 khi nhận cùng một dòng.

In [ ]:
# Mở notebook từ thư mục notebooks/ hay thư mục gốc đều chạy được
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Audio, Image, Markdown, display
pd.set_option('display.max_colwidth', 80)
print('Thư mục làm việc:', ROOT)

## 1. Gửi gì thì nhận gì

In [ ]:
from app.fake_device import FakeDevice

device = FakeDevice()
table = [{'vào': '(khởi động)', 'trả lời': device.boot(), 'đèn': device.state['light'], 'quạt': device.state['fan']}]
inputs = [b'CMD,1,LIGHT_ON\n', b'CMD,2,FAN_ON\r\n', b'CMD,3,LIGHT_ON\n', b'CMD,4,TOGGLE\n',
          b'CMD,abc,FAN_OFF\n', b'CMD,0,FAN_OFF\n', b'HELLO\n', b'X' * 70 + b'\n', b'CMD,5,FAN_OFF\n']
for packet in inputs:
    reply = device.feed(packet)
    shown = packet if len(packet) < 30 else packet[:12] + b'...(%d byte)' % len(packet)
    table.append({'vào': shown, 'trả lời': reply, 'đèn': device.state['light'], 'quạt': device.state['fan']})
pd.DataFrame(table)

Lệnh sai (`TOGGLE`, id không phải số, id bằng 0, dòng rác, dòng dài quá 64 byte) đều bị trả `ERR` và trạng thái đèn/quạt không đổi. Lệnh ON/OFF là đặt trạng thái nên gửi `LIGHT_ON` hai lần thì đèn vẫn chỉ là bật.

## 2. App nói chuyện với thiết bị ảo

In [ ]:
from app.serial_client import open_backend, device_state

client = open_backend('fake')            # giống: python -m app.gui --port fake
for action in ['LIGHT_ON', 'FAN_ON', 'LIGHT_OFF']:
    print(action, '->', client.send(action), device_state(client))
client.close()

## 3. Khi thiết bị gặp sự cố

In [ ]:
from app.fake_device import FakeTransport
from app.serial_client import SerialClient

rows = []
for fault in ['none', 'timeout', 'error', 'reset', 'mismatch']:
    c = SerialClient(transport=FakeTransport(FakeDevice(fault)), timeout=0.3)
    outcome = []
    for attempt in range(2):
        try:
            c.send('FAN_ON'); outcome.append('ACK')
        except Exception as exc:
            outcome.append(f'{type(exc).__name__}: {exc}')
    rows.append({'fault': fault, 'lần 1': outcome[0], 'lần 2': outcome[1]})
pd.DataFrame(rows)

Sau bất kỳ lỗi nào, app sẽ khóa lại và không gửi tiếp cho tới khi kết nối lại (nút "Kết nối lại" trong GUI). Làm vậy là cố ý: khi không chắc thiết bị đang ở trạng thái nào thì không tự gửi lại lệnh.

## 4. Độ trễ gửi lệnh

In [ ]:
import time
from ml import viz

LATENCY_MS = 5     # độ trễ giả lập của thiết bị
c = SerialClient(transport=FakeTransport(FakeDevice(), latency_ms=LATENCY_MS))
samples = []
for i in range(100):
    t = time.perf_counter(); c.send('LIGHT_ON' if i % 2 else 'LIGHT_OFF')
    samples.append((time.perf_counter() - t) * 1000)
viz.latency_hist(samples, f'Từ lúc gửi lệnh tới lúc nhận ACK (thiết bị ảo, trễ {LATENCY_MS} ms)')
plt.show()

## 5. Với board thật

Khi đã có firmware thì bỏ dấu `#` và sửa lại tên cổng COM. Board phải gửi `READY,1` trong vòng 5 giây sau khi mở cổng.

In [ ]:
# client = open_backend('COM5')
# print(client.send('LIGHT_ON'))
# client.close()